In [1]:
import os
import glob
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.interpolate import interp1d
import concurrent.futures
from tqdm import tqdm
import warnings

In [2]:
# --- CONFIGURATION ---
KOI_FILE = "q1_q8_koi_2025.02.03_04.12.15.csv"  # Your catalog file
OUTPUT_DIR = "pipeline_output_v5"
BATCH_SIZE = 10
WORKERS = 3  # Adjust based on your CPU cores
TIMEOUT_SECONDS = 60   # Max time to wait for a single star before skipping

# Image Shapes for ML Team
CNN_GLOBAL_LEN = 5001
CNN_LOCAL_LEN = 501

# Flags
EXPORT_UNFOLDED = True  # Set True to save raw .csvs for Bayesian Signal Detection
INJECT_SYNTHETIC = True # Set True if you have 'batman' installed and want to inject planets

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    if EXPORT_UNFOLDED:
        os.makedirs(f"{OUTPUT_DIR}/bayesian_unfolded")

In [3]:
# --- 1. SYNTHETIC INJECTION (OPTIONAL) ---
def inject_transit(lc, period, t0, planet_radius, star_radius):
    """
    Injects a synthetic transit using the `batman` package.
    [cite_start]References: [cite: 13, 33, 90]
    """
    try:
        import batman
    except ImportError:
        print("Batman package not installed. Skipping injection.")
        return lc

    time = lc.time.value
    
    # Define Batman params
    params = batman.TransitParams()
    params.t0 = t0
    params.per = period
    params.rp = planet_radius / star_radius
    params.a = 15.0 # Semi-major axis (approximation)
    params.inc = 90.0
    params.ecc = 0.0
    params.w = 90.0
    params.u = [0.1, 0.3] # Limb darkening
    params.limb_dark = "quadratic"

    m = batman.TransitModel(params, time)
    flux = m.light_curve(params)
    
    # Inject into existing flux
    lc.flux *= flux
    return lc

In [4]:
# --- UPDATED DATA PROCESSING WITH FLUX ERRORS ---
def build_views(lc, period, t0, duration):
    """
    Generates Global and Local views for both FLUX and FLUX_ERR.
    """
    try:
        # 1. Normalize Flux AND Error
        # We normalize by the median flux. 
        # For error: If flux is divided by M, error must also be divided by M.
        global_median = np.nanmedian(lc.flux.value)
        
        # Normalize in place (lightkurve objects are immutable-ish, so we extract values)
        # Note: We re-normalize locally in the fold, but this gets us to relative units.
        norm_flux = (lc.flux.value - global_median) / global_median
        norm_err = lc.flux_err.value / global_median

        # 2. Fold
        folded = lc.fold(period=period, t0=t0)
        
        # Extract Arrays
        phase = folded.phase.value
        flux = folded.flux.value
        # We need to map the normalized error to the folded indices
        # Since lc.fold() preserves order relative to phase but shuffles time, 
        # we rely on lightkurve's internal handling or just fold the error too if supported.
        # Safest way with Lightkurve:
        flux_err = folded.flux_err.value

        # Re-Normalize Folded Data (Local Median = 0)
        local_median = np.nanmedian(flux)
        flux = (flux - local_median) / local_median
        flux_err = flux_err / local_median # Scale error by same factor
        
        # Sort
        sort_idx = np.argsort(phase)
        phase = phase[sort_idx]
        flux = flux[sort_idx]
        flux_err = flux_err[sort_idx]
        
        # 3. Interpolate (Flux AND Error)
        # We interpolate error linearly. It's an approximation but necessary for fixed-size CNN inputs.
        f_interp = interp1d(phase, flux, kind='linear', fill_value="extrapolate")
        e_interp = interp1d(phase, flux_err, kind='linear', fill_value="extrapolate")
        
        # --- GLOBAL VIEW ---
        x_global = np.linspace(-0.5, 0.5, CNN_GLOBAL_LEN)
        view_global = f_interp(x_global)
        error_global = e_interp(x_global)
        
        # --- LOCAL VIEW ---
        duration_days = duration / 24.0
        duration_phase = duration_days / period
        zoom_width = duration_phase * 2.5 
        
        x_local = np.linspace(-zoom_width, zoom_width, CNN_LOCAL_LEN)
        view_local = f_interp(x_local)
        error_local = e_interp(x_local)
        
        # --- TIME AXIS ---
        time_local_hours = x_local * period * 24.0
        
        return view_global, view_local, error_global, error_local, time_local_hours

    except Exception as e:
        return None, None, None, None, None

def process_star(task):
    row = task['row']
    label = task['label']
    kic = int(row['kepid'])
    do_inject = task['inject']
    
    period = float(row['koi_period'])
    t0 = float(row['koi_time0bk'])
    duration = float(row['koi_duration'])
    
    try:
        # Download & Stitch
        search = lk.search_lightcurve(f"KIC {kic}", author="Kepler", cadence="long")
        if len(search) == 0: return None
        lc_coll = search.download_all()
        if lc_coll is None: return None
         # "Stitch together quarters" to ensure full coverage and better SNR
        lc_raw = lc_coll.stitch()
        
        # Detrend
        # Flatten removes low-frequency stellar variability (rotation, spots)
        # window_length should be larger than transit duration
        lc_clean = lc_raw.remove_nans().flatten(window_length=401).remove_outliers(sigma=5)

        if do_inject:
            # We are turning a False Positive into a Synthetic Planet
            lc_clean = inject_transit(lc_clean, float(row['koi_period']), float(row['koi_time0bk']), 
                                      float(row.get('koi_prad', 2.0)), float(row.get('koi_srad', 1.0)))
            label = 1  # FLIP LABEL TO 1
        
        # Save Unfolded for Bayesian (Now explicitly with Error)
        if EXPORT_UNFOLDED:
            pd.DataFrame({
                'time': lc_clean.time.value,
                'flux': lc_clean.flux.value,
                'flux_err': lc_clean.flux_err.value
            }).to_csv(f"{OUTPUT_DIR}/bayesian_unfolded/kic_{kic}_unfolded.csv", index=False)

        # Generate ML Views (Now includes Error views)
        v_glob, v_loc, e_glob, e_loc, t_loc = build_views(lc_clean, period, t0, duration)
        
        if v_glob is None: return None
        
        return {
            'flux_global': v_glob.astype(np.float32),
            'flux_local': v_loc.astype(np.float32),
            'error_global': e_glob.astype(np.float32),
            'error_local': e_loc.astype(np.float32),
            'time_local': t_loc.astype(np.float32),
            'kic': kic,
            'label': label,
            'period': period,
            't0': t0,
            'duration': duration,
            'transit_depth': float(row.get('koi_depth', 0)),
            'snr': float(row.get('koi_model_snr', 0))
        }
        
    except Exception as e:
        return None

In [5]:
def get_resume_state():
    """Finds completed stars by checking if BOTH metadata AND npz exist."""
    processed_kics = set()
    next_idx = 0
    
    # Look for matching pairs of .npz and _meta.csv
    meta_files = glob.glob(f"{OUTPUT_DIR}/batch_*_meta.csv")
    
    print(f"Scanning {len(meta_files)} batches for existing progress...")
    
    valid_indices = []
    for f in meta_files:
        try:
            # Extract Batch ID
            batch_id = int(os.path.basename(f).split('_')[1])
            npz_path = f"{OUTPUT_DIR}/batch_{batch_id}.npz"
            
            # STRICT CHECK: Must have both files to count as "done"
            if os.path.exists(npz_path):
                df = pd.read_csv(f, usecols=['kic'])
                processed_kics.update(df['kic'].astype(int).tolist())
                valid_indices.append(batch_id)
        except:
            continue
            
    if valid_indices:
        next_idx = max(valid_indices) + 1
        
    return processed_kics, next_idx

def save_batch(data, idx):
    if not data: return
    
    try:
        # 1. Save Arrays (.npz)
        arrays = {k: np.array([d[k] for d in data]) for k in ['flux_global', 'flux_local', 'error_global', 'error_local', 'time_local']}
        np.savez_compressed(f"{OUTPUT_DIR}/batch_{idx}.npz", **arrays)
        
        # 2. Save Metadata (.csv)
        meta_data = [{k: v for k, v in d.items() if k not in arrays} for d in data]
        pd.DataFrame(meta_data).to_csv(f"{OUTPUT_DIR}/batch_{idx}_meta.csv", index=False)
        
        print(f"  [SAVED] Batch {idx} ({len(data)} stars) -> {OUTPUT_DIR}")
    except Exception as e:
        print(f"  [ERROR] Failed to save batch {idx}: {e}")

def merge_metadata():
    """Safely merges only the CSV files into a master index."""
    print("\n--- Finalizing Metadata ---")
    meta_files = sorted(glob.glob(f"{OUTPUT_DIR}/batch_*_meta.csv"))
    
    if not meta_files:
        print("No batches found to merge.")
        return

    all_dfs = []
    for f in tqdm(meta_files, desc="Merging"):
        try:
            df = pd.read_csv(f)
            # Add pointer to the NPZ file
            df['filename'] = os.path.basename(f).replace("_meta.csv", ".npz")
            df['index_in_batch'] = np.arange(len(df))
            all_dfs.append(df)
        except:
            print(f"Skipping corrupt file: {f}")

    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        final_df.to_csv(f"{OUTPUT_DIR}/final_metadata.csv", index=False)
        print(f"SUCCESS: Master Index saved to {OUTPUT_DIR}/final_metadata.csv")
        print(f"Total Processed Stars: {len(final_df)}")
    else:
        print("Merge failed: No valid data found.")

def run_pipeline():
    print("Loading Catalog...")
    df = pd.read_csv(KOI_FILE, comment='#').dropna(subset=['kepid', 'koi_period'])
    
    # Balanced Sampling
    pos = df[df['koi_disposition'] == 'CONFIRMED']
    neg = df[df['koi_disposition'] == 'FALSE POSITIVE']

    # FIX: Define the required variables
    confirmed = pos
    false_pos = neg
    
    print(f"Pool size: {len(confirmed)} Confirmed | {len(false_pos)} False Positives")

    # Goal: 50% Positive (Real + Synthetic), 50% Negative (False Positives)
    n_positive_target = 1000
    n_negative_target = 1000
    
    # Seeded sampling for consistency
    n_real = min(len(confirmed), n_positive_target)
    real_planets = confirmed.sample(n=n_real, random_state=42)
    
    # B. Synthetic Planets (Label 1 from False Positives)
    # Fill the rest of the positive quota with synthetics
    n_synthetic = n_positive_target - n_real
    
    # C. Negatives (Label 0 from False Positives)
    # We need enough False Positives for BOTH the synthetic source AND the negative class
    if len(false_pos) < (n_synthetic + n_negative_target):
        print("Warning: Not enough False Positives for full balance. Reducing dataset size.")
        # Adjust logic or just take what we have
        n_negative_target = len(false_pos) - n_synthetic

    # Shuffle False Positives and split
    fp_shuffled = false_pos.sample(frac=1, random_state=42)
    synthetic_source = fp_shuffled.iloc[:n_synthetic]      # Will be injected (Label 0 -> 1)
    negative_source  = fp_shuffled.iloc[n_synthetic:n_synthetic+n_negative_target] # Keep as Noise (Label 0)

    # 3. Create Tasks
    tasks = []
    
    # Real Planets (No Injection, Label 1)
    for _, r in real_planets.iterrows(): 
        tasks.append({'row': r, 'label': 1, 'inject': False})
        
    # Synthetic Planets (Yes Injection, Label 0 -> 1)
    for _, r in synthetic_source.iterrows(): 
        tasks.append({'row': r, 'label': 0, 'inject': True}) # Label flips in process_star
        
    # False Positives (No Injection, Label 0)
    for _, r in negative_source.iterrows(): 
        tasks.append({'row': r, 'label': 0, 'inject': False})

    print(f"Task Breakdown: {len(real_planets)} Real + {len(synthetic_source)} Synthetic vs {len(negative_source)} Negatives")
    
    # 4. Check Resume Status
    done_kics, batch_idx = get_resume_state()
    tasks_to_run = [t for t in tasks if int(t['row']['kepid']) not in done_kics]
    
    print(f"Remaining Tasks: {len(tasks_to_run)}")
    if not tasks_to_run: return

    # 5. Execute
    current_batch = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
        future_to_task = {executor.submit(process_star, t): t for t in tasks_to_run}
        
        for future in tqdm(concurrent.futures.as_completed(future_to_task), total=len(tasks_to_run)):
            try:
                res = future.result(timeout=TIMEOUT_SECONDS)
                if res: current_batch.append(res)
            except: pass
            
            if len(current_batch) >= BATCH_SIZE:
                save_batch(current_batch, batch_idx)
                current_batch = []
                batch_idx += 1
                
    if current_batch: save_batch(current_batch, batch_idx)

def force_merge():
    print(f"Scanning '{OUTPUT_DIR}' for existing batches...")
    
    # 1. Find all batch metadata files
    meta_files = sorted(glob.glob(f"{OUTPUT_DIR}/batch_*_meta.csv"))
    
    if not meta_files:
        print("No batch files found. Nothing to merge.")
        return

    print(f"Found {len(meta_files)} batches. Merging now...")

    # 2. Read and Combine
    all_meta = []
    total_stars = 0
    
    for f in meta_files:
        try:
            df = pd.read_csv(f)
            
            # CRITICAL: Link this row to its specific .npz file
            # e.g., "batch_0_meta.csv" -> "batch_0.npz"
            batch_filename = os.path.basename(f).replace("_meta.csv", ".npz")
            df['filename'] = batch_filename
            
            # Create a local index (0, 1, 2...) so the loader knows which array to grab
            df['index_in_batch'] = np.arange(len(df))
            
            all_meta.append(df)
            total_stars += len(df)
            print(f"  - Loaded {os.path.basename(f)} ({len(df)} stars)")
            
        except Exception as e:
            print(f"  ⚠️ Skipping corrupt file {f}: {e}")

    # 3. Save Final Index
    if all_meta:
        final_df = pd.concat(all_meta, ignore_index=True)
        final_df.to_csv(f"{OUTPUT_DIR}/final_metadata.csv", index=False)
        
        print("-" * 40)
        print(f"MERGE COMPLETE")
        print(f"Total Stars Secured: {len(final_df)}")
        print(f"  - Planets:         {len(final_df[final_df['label'] == 1])}")
        print(f"  - False Positives: {len(final_df[final_df['label'] == 0])}")
        print(f"Master Index saved to: {f"{OUTPUT_DIR}/final_metadata.csv"}")
        print("-" * 40)
        print("You can now proceed to training your models.")
    else:
        print("Merge failed: valid files found, but dataframes were empty.")

In [ ]:
if __name__ == "__main__":
    run_pipeline()
    merge_metadata()
    force_merge()

Loading Catalog...
Pool size: 2317 Confirmed | 701 False Positives
Task Breakdown: 1000 Real + 0 Synthetic vs 701 Negatives
Scanning 22 batches for existing progress...
Remaining Tasks: 1448


 53%|█████▎    | 765/1448 [14:16<12:45,  1.12s/it]
